# Random Forest classification of physiological conditions

This notebook reproduces the illustrative Random Forest classification analysis reported in the associated manuscript.

The analysis uses the preprocessed **EDA/GSR** and **PPG-derived HR** signals generated by `preprocessing.ipynb`. For each device/placement, both signals are standardized relative to the participant-specific baseline interval, divided into **8-second windows with 50% overlap**, and transformed into time-series features using `tsfresh`.

Features from EDA/GSR and HR are combined for each window. A Random Forest classifier is then evaluated using **leave-one-subject-out cross-validation**, so that all windows from the held-out participant remain exclusively in the test set.

The four classification conditions are:

- `Relax`: the baseline/relaxation interval;
- `Squat`: the squat-test interval;
- `Video 1`;
- `Video 2`.

The purpose of this analysis is illustrative: it assesses whether the recorded physiological signals contain sufficient information to discriminate changes associated with the experimental conditions.


## 1. Imports and configuration

The notebook expects the repository structure used by the other analysis notebooks:

```text
.
├── preprocessing.ipynb
├── classification.ipynb
├── data/
│   └── Stamps/
├── processed_data/
│   └── processed_data.pickle
└── results/
```


In [ ]:
from pathlib import Path
import json
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import LeaveOneGroupOut

from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters



# Paths
PROCESSED_DATA_PATH = Path("processed_data/processed_data.pickle")
STAMPS_DIR = Path("data/Stamps")

FEATURES_DIR = Path("processed_data/classification_features")
OUTPUT_DIR = Path("results/classification")

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



# Devices / placements
DEVICES = [
    "emotibit_volar",
    "emotibit_dorsal",
    "shimmer_wrist",
    "shimmer_fingers",
]

DEVICE_LABELS = {
    "emotibit_volar": "EmotiBit volar",
    "emotibit_dorsal": "EmotiBit dorsal",
    "shimmer_wrist": "Shimmer3 wrist",
    "shimmer_fingers": "Shimmer3 fingers",
}



# Signal-processing branches used in the manuscript
SIGNAL_PATHS = {
    "Gsr": ("IQR", "gaussian"),
    "Hr": ("IQR", "butterworth"),
}



# Classification intervals
CLASS_SEGMENTS = [
    ("relax_start", "relax_end", "Relax"),
    ("squat_test_start", "squat_test_end", "Squat"),
    ("video1_start", "video1_end", "Video 1"),
    ("video2_start", "video2_end", "Video 2"),
]

# Fixed order used in the manuscript confusion matrices.
CLASS_ORDER = [
    "Video 1",
    "Relax",
    "Video 2",
    "Squat",
]



# Feature extraction
WINDOW_SECONDS = 8.0
OVERLAP_FRACTION = 0.50
WINDOW_STEP_SECONDS = WINDOW_SECONDS * (1.0 - OVERLAP_FRACTION)

TSFRESH_N_JOBS = 1
USER_N_JOBS = max(1, (os.cpu_count() or 1) - 1)

# If False, an existing feature CSV is reused.
# On a fresh checkout the CSVs do not exist and are generated automatically.
REBUILD_FEATURES = False


# Random Forest
N_ESTIMATORS = 500
RANDOM_STATE = 42

print(f"Window length: {WINDOW_SECONDS:g} s")
print(f"Window overlap: {OVERLAP_FRACTION:.0%}")
print(f"Window step: {WINDOW_STEP_SECONDS:g} s")
print(f"Random Forest trees: {N_ESTIMATORS}")


## 2. Load the preprocessed physiological data

The classification notebook reads the same `processed_data.pickle` produced by `preprocessing.ipynb`; no additional physiological-data export is required.


In [ ]:
if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"{PROCESSED_DATA_PATH} not found. Run preprocessing.ipynb first."
    )

with PROCESSED_DATA_PATH.open("rb") as file:
    data = pickle.load(file)


def participant_sort_key(value):
    text = str(value)
    return (0, int(text)) if text.isdigit() else (1, text)


missing_devices = [device for device in DEVICES if device not in data]
if missing_devices:
    raise KeyError(
        "Missing devices in processed data: "
        + ", ".join(missing_devices)
    )

participant_sets = [
    set(map(str, data[device].keys()))
    for device in DEVICES
]

participants = sorted(
    set.intersection(*participant_sets),
    key=participant_sort_key,
)

if not participants:
    raise ValueError("No participants are shared by all four device placements.")

print("Participants:", participants)
print("N participants:", len(participants))


## 3. Data-access and baseline-normalization helpers

For each participant, device, and signal, the selected processed signal is standardized using the mean and standard deviation calculated over the participant's baseline (`relax_start` to `relax_end`).

If the baseline standard deviation is zero, only mean centering is applied.


In [ ]:
def load_events(user):
    path = STAMPS_DIR / f"{user}.txt"

    if not path.exists():
        raise FileNotFoundError(f"Event file not found: {path}")

    events = pd.read_csv(
        path,
        header=None,
        names=["Event", "Timestamp"],
    )

    events["Event"] = events["Event"].astype(str).str.strip()
    events["Timestamp"] = pd.to_numeric(
        events["Timestamp"],
        errors="coerce",
    )

    return events.dropna(subset=["Timestamp"])


def event_timestamp(events, event_name):
    values = events.loc[
        events["Event"] == event_name,
        "Timestamp",
    ].to_numpy()

    if len(values) == 0:
        raise KeyError(f"Missing event '{event_name}'")

    return float(values[0])


def nested_get(obj, path):
    current = obj

    for key in path:
        current = current[key]

    return current


def infer_value_column(df):
    if len(df.columns) < 2:
        raise ValueError("Signal dataframe does not contain a value column.")

    candidates = [
        column
        for column in df.columns
        if column != "Timestamp"
        and pd.api.types.is_numeric_dtype(df[column])
    ]

    if not candidates:
        raise ValueError("No numeric signal column found.")

    return candidates[0]


def get_signal_dataframe(device, user, signal):
    user_key = user

    # Preserve compatibility with dictionaries keyed by integer IDs.
    if user_key not in data[device]:
        numeric_user = int(user) if str(user).isdigit() else None
        if numeric_user is not None and numeric_user in data[device]:
            user_key = numeric_user
        else:
            raise KeyError(f"Participant {user} not found for {device}")

    branch = nested_get(
        data[device][user_key][signal],
        SIGNAL_PATHS[signal],
    )

    df = branch["original"]["df"].copy()

    if "Timestamp" not in df.columns:
        raise KeyError(
            f"'Timestamp' missing for {device}/{user}/{signal}"
        )

    value_column = infer_value_column(df)

    output = df[["Timestamp", value_column]].copy()
    output.columns = ["Timestamp", signal]

    output["Timestamp"] = pd.to_numeric(
        output["Timestamp"],
        errors="coerce",
    )
    output[signal] = pd.to_numeric(
        output[signal],
        errors="coerce",
    )

    output = (
        output
        .dropna(subset=["Timestamp", signal])
        .sort_values("Timestamp")
        .reset_index(drop=True)
    )

    return output


def crop_interval(df, start, end):
    return df[
        (df["Timestamp"] >= start)
        & (df["Timestamp"] < end)
    ].copy()


def baseline_standardize(df, signal, baseline_start, baseline_end):
    baseline = crop_interval(
        df,
        baseline_start,
        baseline_end,
    )[signal]

    if baseline.empty:
        raise ValueError(
            f"No {signal} samples found in the baseline interval."
        )

    mean = baseline.mean()
    std = baseline.std()

    normalized = df.copy()

    if np.isfinite(std) and std > 0:
        normalized[signal] = (
            normalized[signal] - mean
        ) / std
    else:
        normalized[signal] = (
            normalized[signal] - mean
        )

    return normalized


## 4. Generate 8-second overlapping windows and extract `tsfresh` features

Each experimental interval is divided into 8-second windows with a 4-second step (50% overlap). A window is retained only when both EDA/GSR and HR contain samples over that interval.

`tsfresh` features are extracted independently from EDA/GSR and HR using `EfficientFCParameters`, then concatenated into one feature vector per window.


In [ ]:
def build_tsfresh_frame(window_df, signal, window_id):
    return (
        window_df
        .rename(columns={signal: "value"})
        .assign(
            id=window_id,
            time=lambda frame: frame["Timestamp"],
            kind=signal,
        )[["id", "time", "kind", "value"]]
    )


def extract_participant_features(device, user):
    events = load_events(user)

    baseline_start = event_timestamp(
        events,
        "relax_start",
    )
    baseline_end = event_timestamp(
        events,
        "relax_end",
    )

    gsr_df = baseline_standardize(
        get_signal_dataframe(
            device,
            user,
            "Gsr",
        ),
        signal="Gsr",
        baseline_start=baseline_start,
        baseline_end=baseline_end,
    )

    hr_df = baseline_standardize(
        get_signal_dataframe(
            device,
            user,
            "Hr",
        ),
        signal="Hr",
        baseline_start=baseline_start,
        baseline_end=baseline_end,
    )

    participant_rows = []

    for start_event, end_event, class_label in CLASS_SEGMENTS:
        segment_start = event_timestamp(
            events,
            start_event,
        )
        segment_end = event_timestamp(
            events,
            end_event,
        )

        window_starts = np.arange(
            segment_start,
            segment_end,
            WINDOW_STEP_SECONDS,
        )

        gsr_windows = []
        hr_windows = []

        for window_start in window_starts:
            window_end = window_start + WINDOW_SECONDS

            if window_end > segment_end:
                continue

            gsr_window = crop_interval(
                gsr_df,
                window_start,
                window_end,
            )

            hr_window = crop_interval(
                hr_df,
                window_start,
                window_end,
            )

            if gsr_window.empty or hr_window.empty:
                continue

            window_id = len(gsr_windows)

            gsr_windows.append(
                build_tsfresh_frame(
                    gsr_window,
                    signal="Gsr",
                    window_id=window_id,
                )
            )

            hr_windows.append(
                build_tsfresh_frame(
                    hr_window,
                    signal="Hr",
                    window_id=window_id,
                )
            )

        if not gsr_windows:
            continue

        gsr_features = extract_features(
            pd.concat(
                gsr_windows,
                axis=0,
                ignore_index=True,
            ),
            column_id="id",
            column_sort="time",
            column_kind="kind",
            column_value="value",
            default_fc_parameters=EfficientFCParameters(),
            n_jobs=TSFRESH_N_JOBS,
            disable_progressbar=True,
        )

        hr_features = extract_features(
            pd.concat(
                hr_windows,
                axis=0,
                ignore_index=True,
            ),
            column_id="id",
            column_sort="time",
            column_kind="kind",
            column_value="value",
            default_fc_parameters=EfficientFCParameters(),
            n_jobs=TSFRESH_N_JOBS,
            disable_progressbar=True,
        )

        common_ids = gsr_features.index.intersection(
            hr_features.index
        )

        for window_id in common_ids:
            row = {
                "Class": class_label,
                "User": str(user),
            }

            row.update(
                gsr_features.loc[
                    window_id
                ].to_dict()
            )

            row.update(
                hr_features.loc[
                    window_id
                ].to_dict()
            )

            participant_rows.append(row)

    return participant_rows


def clean_feature_table(feature_table):
    if feature_table.empty:
        return feature_table

    metadata = feature_table[
        ["Class", "User"]
    ].copy()

    numeric = (
        feature_table
        .drop(
            columns=["Class", "User"],
            errors="ignore",
        )
        .select_dtypes(include=np.number)
    )

    # Keep only features defined for every retained window.
    numeric = numeric.dropna(axis=1)

    # Remove features that are constant over the complete device dataset.
    variable_columns = (
        numeric.var(axis=0) > 0
    )

    numeric = numeric.loc[
        :,
        variable_columns
    ]

    return pd.concat(
        [
            metadata.reset_index(drop=True),
            numeric.reset_index(drop=True),
        ],
        axis=1,
    )


def feature_cache_path(device):
    return FEATURES_DIR / (
        f"{device}_features_"
        f"w{WINDOW_SECONDS:g}_"
        f"overlap{int(OVERLAP_FRACTION * 100)}.csv"
    )


def get_device_features(device):
    cache_path = feature_cache_path(device)

    if (
        cache_path.exists()
        and not REBUILD_FEATURES
    ):
        print(f"Loading cached features: {cache_path}")
        return pd.read_csv(
            cache_path,
            dtype={"User": str},
        )

    print(f"Extracting features for {device}...")

    participant_results = Parallel(
        n_jobs=min(
            USER_N_JOBS,
            len(participants),
        )
    )(
        delayed(extract_participant_features)(
            device,
            user,
        )
        for user in participants
    )

    rows = [
        row
        for user_rows in participant_results
        for row in user_rows
    ]

    feature_table = clean_feature_table(
        pd.DataFrame(rows)
    )

    if feature_table.empty:
        raise ValueError(
            f"No feature rows were generated for {device}."
        )

    feature_table.to_csv(
        cache_path,
        index=False,
    )

    print(
        f"{device}: "
        f"{len(feature_table)} windows, "
        f"{feature_table.shape[1] - 2} features"
    )

    return feature_table


## 5. Build the feature tables

The generated feature tables are cached under `processed_data/classification_features/`. Deleting those files, or setting `REBUILD_FEATURES = True`, forces feature extraction to run again from `processed_data.pickle`.


In [ ]:
features_by_device = {}

feature_summary = []

for device in DEVICES:
    feature_table = get_device_features(
        device
    )

    features_by_device[device] = (
        feature_table
    )

    feature_summary.append(
        {
            "device": device,
            "participants": feature_table["User"].nunique(),
            "windows": len(feature_table),
            "features": feature_table.shape[1] - 2,
        }
    )

feature_summary_df = pd.DataFrame(
    feature_summary
)

display(feature_summary_df)


## 6. Leave-one-subject-out Random Forest

Each fold holds out all windows from exactly one participant. The Random Forest is trained only on windows from the remaining participants.

All extracted EDA/GSR and HR features retained in the device-specific feature table are used as classifier inputs.


In [ ]:
def evaluate_device(device, feature_table):
    y = feature_table["Class"].astype(str)
    groups = feature_table["User"].astype(str)

    X = feature_table.drop(
        columns=["Class", "User"]
    )

    logo = LeaveOneGroupOut()

    predictions = []
    fold_rows = []

    for fold, (
        train_indices,
        test_indices,
    ) in enumerate(
        logo.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):
        X_train = X.iloc[train_indices]
        X_test = X.iloc[test_indices]

        y_train = y.iloc[train_indices]
        y_test = y.iloc[test_indices]

        test_users = sorted(
            groups.iloc[
                test_indices
            ].unique(),
            key=participant_sort_key,
        )

        if len(test_users) != 1:
            raise RuntimeError(
                "Leave-one-subject-out fold contains "
                f"{len(test_users)} test participants."
            )

        model = RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        model.fit(
            X_train,
            y_train,
        )

        y_pred = model.predict(
            X_test
        )

        fold_rows.append(
            {
                "device": device,
                "fold": fold,
                "test_participant": test_users[0],
                "n_test_windows": len(y_test),
                "accuracy": accuracy_score(
                    y_test,
                    y_pred,
                ),
                "precision_weighted": precision_score(
                    y_test,
                    y_pred,
                    average="weighted",
                    zero_division=0,
                ),
                "recall_weighted": recall_score(
                    y_test,
                    y_pred,
                    average="weighted",
                    zero_division=0,
                ),
                "f1_weighted": f1_score(
                    y_test,
                    y_pred,
                    average="weighted",
                    zero_division=0,
                ),
                "f1_macro": f1_score(
                    y_test,
                    y_pred,
                    average="macro",
                    zero_division=0,
                ),
                "balanced_accuracy": balanced_accuracy_score(
                    y_test,
                    y_pred,
                ),
                "mcc": matthews_corrcoef(
                    y_test,
                    y_pred,
                ),
            }
        )

        for participant, true_label, predicted_label in zip(
            groups.iloc[test_indices],
            y_test,
            y_pred,
        ):
            predictions.append(
                {
                    "device": device,
                    "participant": participant,
                    "true_label": true_label,
                    "predicted_label": str(predicted_label),
                }
            )

    prediction_df = pd.DataFrame(
        predictions
    )

    fold_metrics_df = pd.DataFrame(
        fold_rows
    )

    cm_counts = confusion_matrix(
        prediction_df["true_label"],
        prediction_df["predicted_label"],
        labels=CLASS_ORDER,
    )

    cm_normalized = confusion_matrix(
        prediction_df["true_label"],
        prediction_df["predicted_label"],
        labels=CLASS_ORDER,
        normalize="true",
    )

    return {
        "device": device,
        "n_samples": len(feature_table),
        "n_features": X.shape[1],
        "fold_metrics": fold_metrics_df,
        "predictions": prediction_df,
        "cm_counts": cm_counts,
        "cm_normalized": cm_normalized,
    }


## 7. Run the classification and export numerical results


In [ ]:
results_by_device = {}

summary_rows = []
all_predictions = []

for device in DEVICES:
    print(f"Evaluating {DEVICE_LABELS[device]}...")

    result = evaluate_device(
        device,
        features_by_device[device],
    )

    results_by_device[device] = result

    fold_metrics = result[
        "fold_metrics"
    ]

    summary_rows.append(
        {
            "device": device,
            "n_participants": fold_metrics["test_participant"].nunique(),
            "n_windows": result["n_samples"],
            "n_features": result["n_features"],
            "accuracy_mean": fold_metrics["accuracy"].mean(),
            "accuracy_std": fold_metrics["accuracy"].std(ddof=0),
            "f1_weighted_mean": fold_metrics["f1_weighted"].mean(),
            "f1_weighted_std": fold_metrics["f1_weighted"].std(ddof=0),
            "f1_macro_mean": fold_metrics["f1_macro"].mean(),
            "f1_macro_std": fold_metrics["f1_macro"].std(ddof=0),
            "balanced_accuracy_mean": fold_metrics["balanced_accuracy"].mean(),
            "balanced_accuracy_std": fold_metrics["balanced_accuracy"].std(ddof=0),
            "mcc_mean": fold_metrics["mcc"].mean(),
            "mcc_std": fold_metrics["mcc"].std(ddof=0),
        }
    )

    all_predictions.append(
        result["predictions"]
    )

    result["fold_metrics"].to_csv(
        OUTPUT_DIR / f"{device}_fold_metrics.csv",
        index=False,
    )

    counts_df = pd.DataFrame(
        result["cm_counts"],
        index=CLASS_ORDER,
        columns=CLASS_ORDER,
    )

    normalized_df = pd.DataFrame(
        result["cm_normalized"],
        index=CLASS_ORDER,
        columns=CLASS_ORDER,
    )

    counts_df.to_csv(
        OUTPUT_DIR / f"{device}_confusion_counts.csv"
    )

    normalized_df.to_csv(
        OUTPUT_DIR / f"{device}_confusion_normalized.csv"
    )


summary_df = (
    pd.DataFrame(summary_rows)
    .set_index("device")
    .loc[DEVICES]
    .reset_index()
)

summary_df.to_csv(
    OUTPUT_DIR / "summary_metrics.csv",
    index=False,
)

predictions_df = pd.concat(
    all_predictions,
    ignore_index=True,
)

predictions_df.to_csv(
    OUTPUT_DIR / "all_predictions.csv",
    index=False,
)


experiment_config = {
    "devices": DEVICES,
    "class_order": CLASS_ORDER,
    "window_seconds": WINDOW_SECONDS,
    "overlap_fraction": OVERLAP_FRACTION,
    "window_step_seconds": WINDOW_STEP_SECONDS,
    "baseline_interval": [
        "relax_start",
        "relax_end",
    ],
    "signals": [
        "Gsr",
        "Hr",
    ],
    "signal_processing_paths": {
        key: list(value)
        for key, value in SIGNAL_PATHS.items()
    },
    "feature_extraction": "tsfresh EfficientFCParameters",
    "validation": "LeaveOneGroupOut by participant",
    "classifier": {
        "name": "RandomForestClassifier",
        "n_estimators": N_ESTIMATORS,
        "random_state": RANDOM_STATE,
    },
}

with (
    OUTPUT_DIR / "experiment_config.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_config,
        file,
        indent=2,
    )

display(
    summary_df.round(4)
)


## 8. Normalized confusion matrices

The final figure contains one normalized confusion matrix per device/placement. Rows correspond to the true experimental condition and columns to the predicted condition.


In [ ]:
def annotate_confusion_matrix(ax, matrix):
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix[row, column]

            ax.text(
                column,
                row,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if value >= 0.50 else "black",
            )


fig, axes = plt.subplots(
    1,
    len(DEVICES),
    figsize=(14.5, 4.2),
    sharex=True,
    sharey=True,
)

image = None

for index, device in enumerate(DEVICES):
    ax = axes[index]

    matrix = results_by_device[device]["cm_normalized"]

    image = ax.imshow(
        matrix,
        cmap="Blues",
        vmin=0,
        vmax=1,
        interpolation="nearest",
        aspect="equal",
    )

    annotate_confusion_matrix(ax, matrix)

    ax.set_title(
        DEVICE_LABELS[device],
        fontsize=11,
    )

    ax.set_xticks(np.arange(len(CLASS_ORDER)))
    ax.set_xticklabels(
        CLASS_ORDER,
        rotation=45,
        ha="right",
        rotation_mode="anchor",
        fontsize=8,
    )

    ax.set_yticks(np.arange(len(CLASS_ORDER)))

    if index == 0:
        ax.set_yticklabels(
            CLASS_ORDER,
            fontsize=9,
        )
        ax.set_ylabel(
            "Actual",
            fontsize=11,
        )
    else:
        ax.tick_params(
            axis="y",
            labelleft=False,
        )

    ax.set_xlabel(
        "Prediction",
        fontsize=11,
    )

    ax.tick_params(length=0)

# Deja más espacio a la derecha para la colorbar
fig.subplots_adjust(
    left=0.07,
    right=0.87,
    bottom=0.23,
    top=0.82,
    wspace=0.22,
)

# Eje dedicado SOLO a la colorbar
cax = fig.add_axes([0.89, 0.18, 0.012, 0.64])

colorbar = fig.colorbar(image, cax=cax)
colorbar.set_label("Proportion", fontsize=9)

fig.suptitle(
    "Random Forest classification — 8-s windows, 50% overlap",
    fontsize=15,
)

figure_png = OUTPUT_DIR / "confusion_matrices_rf.png"
figure_pdf = OUTPUT_DIR / "confusion_matrices_rf.pdf"

fig.savefig(
    figure_png,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    figure_pdf,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print(f"Saved: {figure_png}")
print(f"Saved: {figure_pdf}")